# Exercise 9: Retrieval Score Analysis

For 10 different queries, retrieve top-10 chunks and analyze the score distribution.

**Goal:** Find a score threshold that separates relevant from irrelevant chunks.


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [10]:
import numpy as np
import statistics

QUERIES_10 = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
    "How do I adjust the engine timing on a Model T?",
    "What tools do I need to remove the cylinder head?",
    "How do I replace the water pump on a Model T?",
    "What is the valve clearance specification for a Model T?",
    "How do I adjust the brakes on a Model T Ford?",
    "What is the procedure for removing and installing the flywheel?",
]

score_data = []

print(f"{'Query':<45} {'#1':>6} {'#5':>6} {'#10':>6} {'Gap(1-2)':>9} {'Spread':>7}")
print("-" * 85)
for q in QUERIES_10:
    results = retrieve(q, top_k=10)
    scores = [s for _, s in results]
    score_data.append(scores)
    gap = scores[0] - scores[1] if len(scores) > 1 else 0
    spread = scores[0] - scores[-1] if len(scores) > 1 else 0
    print(f"{q[:44]:<45} {scores[0]:>6.3f} {scores[4]:>6.3f} {scores[9]:>6.3f} {gap:>9.3f} {spread:>7.3f}")

# Score distribution summary
all_scores_flat = [s for row in score_data for s in row]
print(f"\nOverall score distribution (all queries × top-10):")
print(f"  Min:    {min(all_scores_flat):.3f}")
print(f"  Median: {statistics.median(all_scores_flat):.3f}")
print(f"  Mean:   {sum(all_scores_flat)/len(all_scores_flat):.3f}")
print(f"  Max:    {max(all_scores_flat):.3f}")


Query                                             #1     #5    #10  Gap(1-2)  Spread
-------------------------------------------------------------------------------------
How do I adjust the carburetor on a Model T?   0.640  0.567  0.534     0.063   0.106
What is the correct spark plug gap for a Mod   0.553  0.507  0.491     0.000   0.063
How do I fix a slipping transmission band?     0.601  0.514  0.490     0.000   0.111
What oil should I use in a Model T engine?     0.397  0.376  0.363     0.000   0.034
How do I adjust the engine timing on a Model   0.439  0.436  0.417     0.000   0.022
What tools do I need to remove the cylinder    0.551  0.539  0.499     0.000   0.051
How do I replace the water pump on a Model T   0.355  0.335  0.305     0.005   0.050
What is the valve clearance specification fo   0.585  0.560  0.479     0.000   0.106
How do I adjust the brakes on a Model T Ford   0.498  0.492  0.466     0.000   0.032
What is the procedure for removing and insta   0.541  0.537  0.5

In [11]:
# Experiment: Apply score thresholds
THRESHOLDS = [0.3, 0.4, 0.5, 0.6]

probe_q = "How do I adjust the carburetor on a Model T?"
results_full = retrieve(probe_q, top_k=10)

print(f"Query: '{probe_q}'\n")
for thresh in THRESHOLDS:
    kept = [(c, s) for c, s in results_full if s >= thresh]
    if not kept:
        print(f"  Threshold {thresh}: 0 chunks kept — no context, cannot answer.")
        continue
    ctx = "\n\n---\n\n".join([f"[Score:{s:.3f}]\n{c.text}" for c, s in kept])
    answer = generate_response(
        PROMPT_TEMPLATE.format(context=ctx, question=probe_q)
    )
    print(f"  Threshold {thresh}: {len(kept)} chunks kept")
    print(f"  Answer: {answer[:200]}\n")


Query: 'How do I adjust the carburetor on a Model T?'

  Threshold 0.3: 10 chunks kept
  Answer: To adjust the carburetor on a Model T, follow these steps:

1. **Locate the Adjustment**: Find the adjustment mechanism on the dashboard. This is typically marked with "Adjustment" or similar wording.

  Threshold 0.4: 10 chunks kept
  Answer: To adjust the carburetor on a Model T, you need to follow these steps:

1. **Locate the Adjustment**: Find the adjustment mechanism on the dashboard. This is typically marked with "Adjustment" or simi

  Threshold 0.5: 10 chunks kept
  Answer: To adjust the carburetor on a Model T, you should follow these steps:

1. **Locate the Adjustment**: Find the adjustment mechanism on the dashboard. This is typically indicated by a dial or knob label

  Threshold 0.6: 1 chunks kept
  Answer: To adjust the carburetor on a Model T, you would need to follow these steps:

1. Locate the choke lever (C) at the top of the engine compartment.
2. Turn the choke lever co

## Analysis

**Score patterns observed:**

| Query | #1 score | Gap(1-2) | Spread | Pattern |
|-------|----------|----------|--------|---------|
| Carburetor adjust | 0.640 | **0.063** | 0.106 | Only query with a meaningful gap — relatively clear winner |
| Slipping transmission | 0.601 | 0.000 | 0.111 | High score but no gap — top 2 tied |
| Valve clearance | 0.585 | 0.000 | 0.106 | High score, tightly clustered top ranks |
| Spark plug gap | 0.553 | 0.000 | 0.063 | Moderate, no clear winner |
| Cylinder head tools | 0.551 | 0.000 | 0.051 | Moderate, tight cluster |
| Brakes adjust | 0.498 | 0.000 | 0.032 | Low spread — all 10 chunks nearly identical score |
| Engine timing | 0.439 | 0.000 | 0.022 | Very low spread — ambiguous retrieval |
| Engine oil | 0.397 | 0.000 | 0.034 | Low top score — vocabulary mismatch |
| Water pump | 0.355 | 0.000 | 0.050 | Lowest top score — likely absent from manual |
| Flywheel remove/install | 0.541 | 0.000 | 0.015 | Near-zero spread — all chunks equally matched |

- **Clear winner (large gap):** Only the carburetor query (Gap=0.063). For 9/10 queries Gap=0.000, meaning the top two chunks tied — MiniLM cosine similarity does not produce well-separated single winners on this corpus.
- **Ambiguous clusters (tight spread):** Engine timing (0.022) and flywheel (0.015) have near-zero spread across all 10 chunks — the model cannot distinguish a "best" chunk, which leads to noisy answer content.
- **Low top scores = vocabulary mismatch:** "Water pump" (0.355) and "engine oil" (0.397) either don't appear with those exact terms in the 1919 manual (the manual may say "oiling system" or "cooling apparatus") or the relevant content is too sparse.

**Threshold experiment findings:**

| Threshold | Chunks kept (carburetor Q) | Answer quality |
|-----------|---------------------------|----------------|
| 0.3 | 10 | Same as no threshold — all scored ≥0.53 |
| 0.4 | 10 | Same — threshold too low for this corpus |
| 0.5 | 10 | Same — still all chunks pass; unhelpful |
| 0.6 | 1 | Focused answer from the top chunk only — concise but risks missing context |

**Practical implication:** Hard absolute score thresholds don't work well here. The score floor is ~0.30 and most relevant chunks cluster between 0.50–0.64, so thresholds below 0.60 filter nothing useful. A threshold of 0.60 is over-aggressive, reducing context to a single chunk. **Better alternatives:** (1) relative threshold (keep chunks within 15% of top score); (2) use Gap > 0.05 as a "confident retrieval" flag before applying any threshold; (3) cap at top-k and accept all results, relying on prompt grounding to suppress irrelevant content.

In [12]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
